In [2]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters chromadb pypdf scikit-learn

In [3]:
import os 
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH="/Volumes/Work/Oracle_5_8/gpt-1paper.pdf"

# create the loader object
loader = PyPDFLoader(PDF_PATH)

docs = loader.load()

print(f"Total pages loaded: {len(docs)}")
print(f"Page content with first 500 characters: {docs[0].page_content[:500]}")


Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 67 0 (offset 0)


Total pages loaded: 12
Page content with first 500 characters: Improving Language Understanding
by Generative Pre-Training
Alec Radford
OpenAI
alec@openai.com
Karthik Narasimhan
OpenAI
karthikn@openai.com
Tim Salimans
OpenAI
tim@openai.com
Ilya Sutskever
OpenAI
ilyasu@openai.com
Abstract
Natural language understanding comprises a wide range of diverse tasks such
as textual entailment, question answering, semantic similarity assessment, and
document classiﬁcation. Although large unlabeled text corpora are abundant,
labeled data for learning these speciﬁc tas


In [7]:
# inspect pages

for i,doc in enumerate(docs):
    preview = doc.page_content[:100].replace("\n", " ")
    print(f"{i} | {len(doc.page_content)} | preview: {preview}")

0 | 3344 | preview: Improving Language Understanding by Generative Pre-Training Alec Radford OpenAI alec@openai.com Kart
1 | 4886 | preview: In this paper, we explore a semi-supervised approach for language understanding tasks using a combin
2 | 3255 | preview: pre-trained language or machine translation model as auxiliary features while training a supervised 
3 | 2882 | preview: Figure 1:(left)Transformer architecture and training objectives used in this work.(right)Input trans
4 | 4150 | preview: Table 1: A list of the different tasks and datasets used in our experiments. Task Datasets Natural l
5 | 3337 | preview: Table 2: Experimental results on natural language inference tasks, comparing our model with current 
6 | 2265 | preview: Table 4: Semantic similarity and classiﬁcation results, comparing our model with current state-of-th
7 | 4259 | preview: Table 5: Analysis of various model ablations on different tasks. Avg. score is a unweighted average 
8 | 3806 | preview: [2]J. L. Ba,

In [8]:
# text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 80
)

chunks = text_splitter.split_documents(docs)
print(f"Chunks created: {len(chunks)}")
# sample chunk
print(f"Sample chunk content with first 500 characters: {chunks[0].page_content[:100]}")

Chunks created: 276
Sample chunk content with first 500 characters: Improving Language Understanding
by Generative Pre-Training
Alec Radford
OpenAI
alec@openai.com
Kart


In [11]:
# inspect chunks

for i in range(4):
    print(f"Chunk : {i}")
    print(f"Metadata : {chunks[i].metadata}")
    print(f"Length : {len(chunks[i].page_content)}")
    print(f"Content : {chunks[i].page_content[:100]}")
    print("--------------------------------------------------")

Chunk : 0
Metadata : {'producer': 'macOS Version 26.6.2 (Build 25G83) Quartz PDFContext', 'creator': 'Safari', 'creationdate': "D:20261006050416Z00'00'", 'moddate': "D:20261006050416Z00'00'", 'author': 'Chirantan Lonkar', 'title': 'https:/cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper', 'source': '/Volumes/Work/Oracle_5_8/gpt-1paper.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}
Length : 198
Content : Improving Language Understanding
by Generative Pre-Training
Alec Radford
OpenAI
alec@openai.com
Kart
--------------------------------------------------
Chunk : 1
Metadata : {'producer': 'macOS Version 26.6.2 (Build 25G83) Quartz PDFContext', 'creator': 'Safari', 'creationdate': "D:20261006050416Z00'00'", 'moddate': "D:20261006050416Z00'00'", 'author': 'Chirantan Lonkar', 'title': 'https:/cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper', 'source': '/Volumes/Work/Oracle_5_8/gpt-1paper.pdf', 'total_pages': 12, 'page

In [13]:
# embeddings 

from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

#embed single chunk 
sample_chunk = chunks[0].page_content
sample_vector = embedding_model.embed_query(sample_chunk)
print(f"Sample vector text: {sample_chunk[:100]}")
print(f"Sample chunk vector length: {len(sample_vector)}")
print(f"First 10 dimensions of the sample vector: {sample_vector[:10]}")

Sample vector text: Improving Language Understanding
by Generative Pre-Training
Alec Radford
OpenAI
alec@openai.com
Kart
Sample chunk vector length: 3072
First 10 dimensions of the sample vector: [0.0220947265625, 0.017547607421875, -0.0181884765625, -0.0290985107421875, -0.0093841552734375, 0.03411865234375, -0.01043701171875, 0.033355712890625, -0.017730712890625, -0.01611328125]


In [17]:
# store vectors in a vector database

from langchain_chroma import Chroma

print(f"Embedding {len(chunks)} chunks into vector database")

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "./oracle_rag_db",
    collection_name = "gpt_1_paper"
)

total = vectorstore._collection.count()
print(f"Total Chunks stored in the database: {total}")

Embedding 276 chunks into vector database
Total Chunks stored in the database: 276


In [18]:
# peek into the database

collection_data = vectorstore._collection.get(
    include = ["metadatas", "documents", "embeddings"],
)

ids = collection_data['ids']
embeddings = collection_data['embeddings']
metadatas = collection_data['metadatas']
documents = collection_data['documents']  

for i in range(5):
    print(f"Chunk ID: {ids[i]}")
    print(f"Metadata: {metadatas[i]}")
    print(f"Chunk content : {documents[i][:100]}")  # Print first 100 characters of the document
    print(f"Embedding length: {len(embeddings[i])}")
    print(f"First 10 dimensions of the embedding: {embeddings[i][:10]}")
    print("--------------------------------------------------")

Chunk ID: 570b1fc7-b835-4eb2-9913-acc06311e148
Metadata: {'title': 'https:/cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper', 'total_pages': 12, 'producer': 'macOS Version 26.6.2 (Build 25G83) Quartz PDFContext', 'author': 'Chirantan Lonkar', 'creationdate': "D:20261006050416Z00'00'", 'creator': 'Safari', 'moddate': "D:20261006050416Z00'00'", 'page_label': '1', 'page': 0, 'source': '/Volumes/Work/Oracle_5_8/gpt-1paper.pdf'}
Chunk content : Improving Language Understanding
by Generative Pre-Training
Alec Radford
OpenAI
alec@openai.com
Kart
Embedding length: 3072
First 10 dimensions of the embedding: [ 0.02207947  0.01756287 -0.01818848 -0.02903748 -0.00940704  0.03408813
 -0.01047516  0.03338623 -0.01776123 -0.01609802]
--------------------------------------------------
Chunk ID: 9dadac85-d142-4aac-b3f0-843856ee1606
Metadata: {'source': '/Volumes/Work/Oracle_5_8/gpt-1paper.pdf', 'moddate': "D:20261006050416Z00'00'", 'total_pages': 12, 'title': 'https:/cd

In [19]:
# dummy retireval function with cosine similarity
def retrieve_with_scores(query:str, k:int=3):
    "retrieve the top k chinks with cosine scores"
    results= vectorstore.similarity_search_with_relevance_scores(query,k=k)
    print(f"Query: {query}")
    print(f"Top {k} results: \n")
    for i,(doc,score) in enumerate(results):
        print(f"Rank {i+1} | score: {score : .4f} | Page : {doc.metadata.get('page', '?')}")
        print(f"{doc.page_content[:200]}")  # Print first 200 characters of the document
        print()
    return results

In [20]:
result = retrieve_with_scores("Which datasets were used for Natural language inference? ")

print(result)

Query: Which datasets were used for Natural language inference? 
Top 3 results: 

Rank 1 | score:  0.4759 | Page : 8
challenge. InTAC, 2009.
[5]S. R. Bowman, G. Angeli, C. Potts, and C. D. Manning. A large annotated corpus for learning natural
language inference.EMNLP, 2015.

Rank 2 | score:  0.4753 | Page : 4
provides an overview of all the tasks and datasets.
Natural Language InferenceThe task of natural language inference (NLI), also known as recog-

Rank 3 | score:  0.4730 | Page : 4
Table 1: A list of the different tasks and datasets used in our experiments.
Task Datasets
Natural language inference SNLI [5], MultiNLI [66], Question NLI [64], RTE [4], SciTail [25]

[(Document(id='ddc6141b-ff9c-4996-b44b-9a24329534a0', metadata={'total_pages': 12, 'producer': 'macOS Version 26.6.2 (Build 25G83) Quartz PDFContext', 'creator': 'Safari', 'title': 'https:/cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper', 'author': 'Chirantan Lonkar', 'page': 8, 'moddate

In [ ]:
# RAG

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o",temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def rag_answer(query:str, show_sources:bool=True) -> str:
    # step 1: retrieve relevant chunks
    retrieved = retriever.invoke(query)
    # step 2: build context 
    context = "\n\n".join(
        f"[Page {doc.metadata.get('page', '?')}]\n{doc.page_content}" 
        for doc in retrieved
    )
    # step 3 : build prompt
    messages = [
        SystemMessage(content=(
            "You are a precise research paper assistant. "
            "Answer the question using ONLY the provided context from the GPT-1 paper. "
            "If the answer is not in the context, say: 'This is not covered in the retrieved sections.' "
            "Be specific, cite page numbers where possible, and do not hallucinate."
        )),
        HumanMessage(content=f"Context from GPT-1 paper:\n{context}\n\nQuestion: {query}")
    ]
    
    # strep 4 :call the LLM
    response = llm.invoke(messages)
    
    if show_sources:
        print("Sources used for the answer:\n")
        for i,doc in enumerate(retrieved):
            print(f"Source {i+1} | Page: {doc.metadata.get('page', '?')}")
            print(f"{doc.page_content[:200]}")  # Print first 200 characters of the document
            print("--------------------------------------------------")
    return response.content